In [1]:
# ============================================================
# PROJECT 2: SPOTIFY SONGS' GENRE SEGMENTATION
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

# Plot settings
sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")
# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "spotify dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
# ============================================================
# 3. DISPLAY DATASET
# ============================================================

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("\nDataset Information:")
df.info()
# ============================================================
# 5. COLUMN NAMES
# ============================================================

print("\nColumn Names:")

for i, column in enumerate(df.columns, start=1):
    print(i, "-", column)
# ============================================================
# 6. CHECK DUPLICATES
# ============================================================

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")

print("Dataset shape after duplicate removal:", df.shape)
# ============================================================
# 7. CHECK MISSING VALUES
# ============================================================

missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Column": missing_values.index,
    "Missing Values": missing_values.values
})

print("\nMissing Values:")
display(missing_table)
# ============================================================
# MISSING VALUE VISUALIZATION
# ============================================================

plt.figure(figsize=(12, 6))

missing_values[missing_values > 0].sort_values(ascending=False).plot(
    kind="bar"
)

plt.title("Missing Values by Column")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# ============================================================
# 8. HANDLE MISSING VALUES
# ============================================================

# Numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Categorical/object columns
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")

print("Missing values handled successfully.")

print("Total remaining missing values:",
      df.isnull().sum().sum())
# ============================================================
# 9. BASIC STATISTICS
# ============================================================

print("Statistical Summary:")

display(df.describe())
# ============================================================
# 10. GENRE ANALYSIS
# ============================================================

print("Playlist Genres:")

genres = df["playlist_genre"].unique()

for genre in genres:
    print("-", genre)

print("\nTotal genres:", df["playlist_genre"].nunique())
# ============================================================
# 11. GENRE DISTRIBUTION
# ============================================================

genre_counts = df["playlist_genre"].value_counts()

print("Number of songs per genre:")
display(genre_counts)
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="playlist_genre",
    order=genre_counts.index
)

plt.title("Distribution of Spotify Songs by Genre")
plt.xlabel("Playlist Genre")
plt.ylabel("Number of Songs")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()
# ============================================================
# 12. PLAYLIST SUBGENRE
# ============================================================

subgenre_counts = df["playlist_subgenre"].value_counts()

print("Top playlist subgenres:")
display(subgenre_counts.head(20))
plt.figure(figsize=(12, 8))

sns.barplot(
    x=subgenre_counts.head(15).values,
    y=subgenre_counts.head(15).index
)

plt.title("Top 15 Playlist Subgenres")
plt.xlabel("Number of Songs")
plt.ylabel("Playlist Subgenre")

plt.tight_layout()
plt.show()
# ============================================================
# 13. TOP ARTISTS
# ============================================================

top_artists = df["track_artist"].value_counts().head(15)

print("Top 15 Artists:")
display(top_artists)
plt.figure(figsize=(12, 7))

sns.barplot(
    x=top_artists.values,
    y=top_artists.index
)

plt.title("Top 15 Artists in the Dataset")
plt.xlabel("Number of Songs")
plt.ylabel("Artist")

plt.tight_layout()
plt.show()
# ============================================================
# 14. TRACK POPULARITY
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["track_popularity"],
    bins=20
)

plt.title("Distribution of Track Popularity")
plt.xlabel("Track Popularity")
plt.ylabel("Number of Songs")

plt.tight_layout()
plt.show()
# ============================================================
# 15. AUDIO FEATURES
# ============================================================

audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

print("Audio features used:")
for feature in audio_features:
    print("-", feature)
# ============================================================
# 16. AUDIO FEATURE DISTRIBUTIONS
# ============================================================

for feature in audio_features:

    plt.figure(figsize=(8, 5))

    plt.hist(
        df[feature],
        bins=30
    )

    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.show()
    # ============================================================
# 17. BOXPLOTS
# ============================================================

plt.figure(figsize=(14, 7))

sns.boxplot(
    data=df[audio_features]
)

plt.title("Boxplot of Spotify Audio Features")
plt.xlabel("Audio Features")
plt.ylabel("Values")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()
# ============================================================
# 18. DANCEABILITY VS ENERGY
# ============================================================

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df,
    x="danceability",
    y="energy",
    hue="playlist_genre",
    alpha=0.5
)

plt.title("Danceability vs Energy by Genre")
plt.xlabel("Danceability")
plt.ylabel("Energy")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()
# ============================================================
# 19. ENERGY VS POPULARITY
# ============================================================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="energy",
    y="track_popularity",
    alpha=0.5
)

plt.title("Energy vs Track Popularity")
plt.xlabel("Energy")
plt.ylabel("Track Popularity")

plt.tight_layout()
plt.show()
# ============================================================
# 20. ACOUSTICNESS VS ENERGY
# ============================================================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="acousticness",
    y="energy",
    hue="playlist_genre",
    alpha=0.5
)

plt.title("Acousticness vs Energy")
plt.xlabel("Acousticness")
plt.ylabel("Energy")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()
# ============================================================
# 21. GENRE-WISE AUDIO FEATURES
# ============================================================

genre_audio_means = df.groupby(
    "playlist_genre"
)[audio_features].mean()

print("Average audio features by genre:")

display(genre_audio_means)
# ============================================================
# 22. GENRE AUDIO FEATURE HEATMAP
# ============================================================

plt.figure(figsize=(14, 7))

sns.heatmap(
    genre_audio_means,
    annot=True,
    fmt=".2f",
    cmap="viridis"
)

plt.title("Average Audio Features by Playlist Genre")
plt.xlabel("Audio Features")
plt.ylabel("Genre")

plt.tight_layout()
plt.show()
# ============================================================
# 23. CORRELATION MATRIX
# ============================================================

correlation_features = audio_features + ["track_popularity"]

correlation_matrix = df[
    correlation_features
].corr()

print("Correlation Matrix:")

display(correlation_matrix)
plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix of Spotify Features")

plt.tight_layout()
plt.show()
# ============================================================
# 24. PREPARE DATA FOR CLUSTERING
# ============================================================

X = df[audio_features].copy()

print("Clustering dataset shape:", X.shape)

display(X.head())
# ============================================================
# 25. STANDARDIZATION
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Features standardized successfully!")

print("Scaled data shape:", X_scaled.shape)
# ============================================================
# 26. ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(model.inertia_)
    plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.title("Elbow Method for Finding Optimal Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")

plt.xticks(k_values)

plt.tight_layout()
plt.show()
# ============================================================
# 27. SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"K = {k}  -->  Silhouette Score = {score:.4f}"
    )
    plt.figure(figsize=(10, 6))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.title("Silhouette Score for Different Numbers of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")

plt.xticks(range(2, 11))

plt.tight_layout()
plt.show()
# ============================================================
# 28. SELECT BEST K
# ============================================================

best_k = range(2, 11)[
    np.argmax(silhouette_scores)
]

print("Best number of clusters based on Silhouette Score:",
      best_k)

print(
    "Best Silhouette Score:",
    max(silhouette_scores)
)
# ============================================================
# 29. FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = cluster_labels

print("K-Means clustering completed successfully!")
# ============================================================
# 30. CLUSTER RESULTS
# ============================================================

result_columns = [
    "track_name",
    "track_artist",
    "playlist_name",
    "playlist_genre",
    "playlist_subgenre",
    "Cluster"
]

display(
    df[result_columns].head(20)
)
# ============================================================
# 31. CLUSTER SIZE
# ============================================================

cluster_counts = df["Cluster"].value_counts().sort_index()

print("Number of songs in each cluster:")

display(cluster_counts)
plt.figure(figsize=(9, 6))

sns.barplot(
    x=cluster_counts.index.astype(str),
    y=cluster_counts.values
)

plt.title("Number of Songs in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Songs")

plt.tight_layout()
plt.show()
# ============================================================
# 32. CLUSTER AUDIO FEATURE ANALYSIS
# ============================================================

cluster_analysis = df.groupby(
    "Cluster"
)[audio_features].mean()

print("Average audio features for each cluster:")

display(cluster_analysis)
# ============================================================
# 33. CLUSTER HEATMAP
# ============================================================

plt.figure(figsize=(14, 7))

sns.heatmap(
    cluster_analysis,
    annot=True,
    fmt=".2f",
    cmap="viridis"
)

plt.title("Average Audio Features Across Clusters")
plt.xlabel("Audio Features")
plt.ylabel("Cluster")

plt.tight_layout()
plt.show()
# ============================================================
# 34. CLUSTER VS GENRE
# ============================================================

cluster_genre = pd.crosstab(
    df["Cluster"],
    df["playlist_genre"]
)

print("Cluster vs Playlist Genre:")

display(cluster_genre)
plt.figure(figsize=(10, 7))

sns.heatmap(
    cluster_genre,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Clusters According to Playlist Genre")
plt.xlabel("Playlist Genre")
plt.ylabel("Cluster")

plt.tight_layout()
plt.show()
# ============================================================
# 35. CLUSTER VS PLAYLIST NAME
# ============================================================

top_playlists = df[
    "playlist_name"
].value_counts().head(15).index

playlist_cluster_data = df[
    df["playlist_name"].isin(top_playlists)
]

cluster_playlist = pd.crosstab(
    playlist_cluster_data["Cluster"],
    playlist_cluster_data["playlist_name"]
)

print("Cluster vs Top Playlist Names:")

display(cluster_playlist)
plt.figure(figsize=(16, 8))

sns.heatmap(
    cluster_playlist,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.title("Clusters According to Top Playlist Names")
plt.xlabel("Playlist Name")
plt.ylabel("Cluster")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()
# ============================================================
# 36. GENRE PERCENTAGE WITHIN CLUSTERS
# ============================================================

cluster_genre_percentage = pd.crosstab(
    df["Cluster"],
    df["playlist_genre"],
    normalize="index"
) * 100

print("Genre percentage within each cluster:")

display(
    cluster_genre_percentage.round(2)
)
# ============================================================
# 37. DOMINANT GENRE
# ============================================================

dominant_genre = (
    cluster_genre_percentage
    .idxmax(axis=1)
)

print("Dominant genre in each cluster:")

for cluster, genre in dominant_genre.items():
    print(f"Cluster {cluster}: {genre}")
# ============================================================
# 38. PCA
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = df["Cluster"].values

print("PCA completed.")

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_
)
# ============================================================
# 39. PCA CLUSTER VISUALIZATION
# ============================================================

plt.figure(figsize=(11, 8))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10",
    alpha=0.6
)

plt.title("Spotify Songs Segmentation using K-Means Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()
# ============================================================
# 40. CLUSTER CENTERS
# ============================================================

cluster_centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=audio_features
)

cluster_centers.index.name = "Cluster"

print("Cluster Centers:")

display(
    cluster_centers.round(3)
)
# ============================================================
# 41. SAMPLE SONGS FROM EACH CLUSTER
# ============================================================

for cluster in sorted(df["Cluster"].unique()):

    print("\n" + "=" * 70)
    print(f"CLUSTER {cluster}")
    print("=" * 70)

    songs = df[
        df["Cluster"] == cluster
    ][
        [
            "track_name",
            "track_artist",
            "playlist_genre",
            "playlist_subgenre"
        ]
    ].head(10)

    display(songs)
# ============================================================
# 42. SONG RECOMMENDATION FUNCTION
# ============================================================

def recommend_songs(song_name, number_of_recommendations=5):

    # Find the song
    matches = df[
        df["track_name"]
        .str.lower()
        .str.strip()
        == song_name.lower().strip()
    ]

    if matches.empty:
        print("Song not found in the dataset.")
        return None

    # Get the song's cluster
    song_cluster = matches.iloc[0]["Cluster"]

    print(
        f"'{matches.iloc[0]['track_name']}' "
        f"belongs to Cluster {song_cluster}"
    )

    # Get songs from same cluster
    recommendations = df[
        (df["Cluster"] == song_cluster) &
        (
            df["track_name"].str.lower()
            != song_name.lower()
        )
    ]

    recommendations = recommendations[
        [
            "track_name",
            "track_artist",
            "playlist_genre",
            "playlist_subgenre",
            "track_popularity",
            "Cluster"
        ]
    ].sort_values(
        by="track_popularity",
        ascending=False
    )

    return recommendations.head(
        number_of_recommendations
    )
# ============================================================
# 43. SAMPLE SONG NAMES
# ============================================================

print(
    df["track_name"]
    .dropna()
    .head(20)
    .tolist()
)
recommend_songs(
    "I Don't Care (with Justin Bieber) - Loud Luxury Remix",
    5
)
# ============================================================
# 44. SAVE FINAL DATASET
# ============================================================

output_file = "spotify_clustered_dataset.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    f"Clustered dataset saved as: {output_file}"
)
# ============================================================
# 45. SAVE MACHINE LEARNING MODEL
# ============================================================

joblib.dump(
    kmeans,
    "spotify_kmeans_model.pkl"
)

joblib.dump(
    scaler,
    "spotify_scaler.pkl"
)

joblib.dump(
    pca,
    "spotify_pca.pkl"
)

print("K-Means model saved.")
print("Scaler saved.")
print("PCA model saved.")
# ============================================================
# 46. FINAL PROJECT SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("        SPOTIFY SONGS' GENRE SEGMENTATION")
print("=" * 60)

print("\nDataset Information")
print("-------------------")
print("Total songs:", len(df))
print("Total columns:", df.shape[1])
print("Number of genres:", df["playlist_genre"].nunique())
print("Number of subgenres:", df["playlist_subgenre"].nunique())

print("\nClustering Information")
print("----------------------")
print("Number of clusters:", best_k)

print(
    "Best silhouette score:",
    round(max(silhouette_scores), 4)
)

print("\nCluster Distribution")
print("--------------------")

for cluster, count in cluster_counts.items():
    print(
        f"Cluster {cluster}: {count} songs"
    )

print("\nDominant Genre of Each Cluster")
print("------------------------------")

for cluster, genre in dominant_genre.items():
    print(
        f"Cluster {cluster}: {genre}"
    )

print("\nProject completed successfully!")


Libraries imported successfully!


FileNotFoundError: [Errno 2] No such file or directory: 'spotify dataset.csv'